In [1]:
import torch
import numpy as np


In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

get_device = get_device()
get_device

device(type='cuda')

In [3]:
import pandas as pd

try: # KAGGLE
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
except: 
    try: # COLAB
        df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
    except: # LOCAL
        df = pd.read_csv('../data/train.csv')
        test_df = pd.read_csv('../data/test.csv')
        
df.shape, test_df.shape

((2000, 8), (500, 7))

set seed

In [4]:
from transformers import set_seed

set_seed(42)

# Config

In [5]:
MODEL_NAME = "microsoft/deberta-v3-base"
OPTIONS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 3

---

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from datasets import Dataset


# found out this model is good for mcq tasks 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

label2id = {opt: i for i, opt in enumerate(OPTIONS)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias 

# Preprocess function

In [ ]:

def preprocess_function(dataset):
    import itertools
    options = ['A', 'B', 'C', 'D', 'E']
    
    batch_size = len(dataset['prompt'])
    first_sentences = [[question] * 5 for question in dataset['prompt']]
    second_sentences = [[dataset[option][i] for option in options] for i in range(batch_size)]

    # flattening the list to put into tokenizer
    first_sentences = list(itertools.chain.from_iterable(first_sentences))
    second_sentences = list(itertools.chain.from_iterable(second_sentences))

    output = tokenizer(first_sentences, second_sentences, return_tensors="pt", padding='max_length', truncation=True, max_length=256)
    ## FOR THIS PROBLEM THE MODEL NEEDS IN THIS SHAPE : BATCH_SIZE, OPTIONS, MAX_LENGTH
    # have to reshape it back in batch_size, options, max_length
    for key in output:
        output[key] = output[key].view(batch_size, 5, 256)

    # ALSO HAVE TO CREATE THE LABELS FOR USING TRAINER
    if 'answer' in dataset and dataset['answer'] is not None:
        mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        output['labels'] = torch.tensor(
            [mapping[ans] for ans in dataset['answer']], 
            dtype=torch.int
        ) 

    return output


# Dataset load

In [8]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

encoded_train = split_dataset["train"].map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names,
)

encoded_eval = split_dataset["test"].map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names,
)


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

---
testing the modules

---

# Training

In [9]:
# import torch
# import gc

# del model

# # 2. Force Python garbage collection
# gc.collect()

# # 3. Release unallocated cached memory back to the GPU driver
# torch.cuda.empty_cache()

In [ ]:

# Training configuration
training_args = TrainingArguments(
    output_dir="./mcq_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,
)

# Pass eval_dataset to Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    processing_class=tokenizer,
)
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# torch.cuda.device_count(), torch.cuda.get_device_name(0)

(1, 'Tesla T4')

# Test predictions 


In [ ]:
test_df.head(5)

In [ ]:
test_data = Dataset.from_pandas(test_df)
print(type(test_data), test_data.shape)
encoded_test = test_data.map(
    preprocess_function,
    batched=True,
    batch_size=100,
    remove_columns=test_data.column_names,
    load_from_cache_file=False
)

encoded_test.shape


<class 'datasets.arrow_dataset.Dataset'> (500, 7)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


(500, 3)


In [ ]:
# generate the preds

preds = trainer.predict(encoded_test)
raw_logits = preds.predictions
top_3_indices = torch.topk(torch.tensor(raw_logits), k=3, dim=-1).indices.numpy()


NameError: name 'id2label' is not defined

In [ ]:
id2label = {v: k for k, v in label2id.items()}

top_3_predictions = [
    [id2label[idx] for idx in row] 
    for row in top_3_indices
]
print(raw_logits.shape, top_3_indices.shape, top_3_predictions)


(500, 5) (500, 3) [['A', 'E', 'D'], ['B', 'C', 'D'], ['B', 'E', 'D'], ['E', 'C', 'D'], ['C', 'D', 'A'], ['D', 'C', 'A'], ['E', 'C', 'D'], ['B', 'E', 'C'], ['C', 'D', 'E'], ['B', 'E', 'C'], ['A', 'E', 'D'], ['D', 'E', 'B'], ['C', 'A', 'B'], ['C', 'B', 'E'], ['E', 'D', 'B'], ['A', 'C', 'D'], ['E', 'D', 'B'], ['B', 'D', 'E'], ['A', 'C', 'E'], ['D', 'B', 'C'], ['B', 'D', 'C'], ['D', 'C', 'E'], ['D', 'E', 'A'], ['E', 'C', 'D'], ['D', 'C', 'B'], ['E', 'D', 'A'], ['E', 'C', 'D'], ['C', 'E', 'A'], ['B', 'A', 'E'], ['A', 'E', 'D'], ['E', 'C', 'D'], ['C', 'B', 'D'], ['D', 'E', 'C'], ['C', 'E', 'D'], ['A', 'C', 'D'], ['C', 'A', 'B'], ['C', 'E', 'D'], ['C', 'A', 'B'], ['B', 'C', 'D'], ['C', 'E', 'A'], ['A', 'B', 'E'], ['E', 'D', 'A'], ['B', 'C', 'E'], ['A', 'B', 'E'], ['D', 'B', 'E'], ['B', 'A', 'C'], ['D', 'B', 'E'], ['A', 'B', 'D'], ['B', 'E', 'A'], ['C', 'D', 'B'], ['D', 'E', 'A'], ['A', 'C', 'D'], ['C', 'D', 'B'], ['C', 'D', 'A'], ['B', 'C', 'D'], ['B', 'E', 'C'], ['B', 'C', 'A'], ['A', 'D', '

In [ ]:
test_df["prediction"] = [" ".join(row) for row in top_3_predictions]

submission_df = test_df[["id", "prediction"]]
submission_df.rename(columns={"id": "Id", "prediction": "Prediction"}, inplace=True)
submission_df.to_csv("submission.csv", index=False)


/tmp/ipykernel_699/825687279.py:4: SettingWithCopyWarning:
A value is trying to be set on a copy of a slice from a DataFrame
See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission_df.rename(columns={"id": "Id", "prediction": "Prediction"}, inplace=True)
